# Notebook 04 — Panel Assembly & Exploratory Data Analysis

This notebook:
1. Merges all features into the final **panel dataset** (20 regions × 11 years)
2. Runs comprehensive **EDA** — summary statistics, correlations, distributions
3. Creates exploratory visualizations

**Prerequisites:** Notebooks 01–03 complete (all interim parquets exist).

In [ ]:
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.utils.config import load_config, get_path
from src.utils.io import load_dataframe, save_dataframe

cfg = load_config()
print("Ready.")

---
## 1. Assemble Panel Dataset

Merge: heatwave metrics + mortality + RSVI + population → single panel with interaction terms.

In [ ]:
from src.analysis.panel_dataset import merge_panel_components, add_derived_variables, load_population

interim_dir = get_path(cfg, "interim_data")

hw   = load_dataframe(interim_dir / "heatwave_metrics.parquet")
mort = load_dataframe(interim_dir / "mortality_processed.parquet")
rsvi = load_dataframe(interim_dir / "rsvi.parquet")
pop  = load_population(cfg)

print("Heatwave:", hw.shape)
print("Mortality:", mort.shape)
print("RSVI:", rsvi.shape)
print("Population:", pop.shape)

In [ ]:
panel = merge_panel_components(mort, hw, rsvi, pop)
panel = add_derived_variables(panel)

print("Panel shape:", panel.shape)
print("Regions:", panel["nuts2_code"].nunique())
print("Years:", sorted(panel["year"].unique()))
print("
Columns:")
for col in panel.columns:
    missing = panel[col].isna().sum()
    flag = " ← missing!" if missing > 0 else ""
    print(f"  {col:<35s} {str(panel[col].dtype):<12s} nulls={missing}{flag}")

In [ ]:
# Save panel
processed_dir = get_path(cfg, "processed_data")
save_dataframe(panel, processed_dir / "panel_dataset.csv", index=False)
save_dataframe(panel, processed_dir / "panel_dataset.parquet", index=False)
print("Panel saved to data/processed/")

---
## 2. Summary Statistics

In [ ]:
from src.analysis.eda import summary_statistics, summary_by_region

stats = summary_statistics(panel)
print("Overall Summary Statistics:")
display(stats.round(2))

In [ ]:
regional = summary_by_region(panel)
print("Regional Summary (means):")
display(regional.round(2))

---
## 3. Correlation Analysis

In [ ]:
from src.analysis.eda import correlation_matrix

fig_dir = get_path(cfg, "figures")
fig_dir.mkdir(parents=True, exist_ok=True)

key_vars = ["mortality_rate", "hw_days", "hw_intensity", "summer_tmax_anomaly", "rsvi"]
available = [v for v in key_vars if v in panel.columns]

corr = panel[available].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.5, ax=ax)
ax.set_title("Key Variable Correlations", fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig(fig_dir / "correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 4. Time Series — Heatwave Trends

In [ ]:
national = panel.groupby("year")[["hw_days", "hw_events", "summer_tmax_mean"]].mean()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(national.index, national["hw_days"], color="#e74c3c", alpha=0.8)
axes[0].set_title("Mean Heatwave Days per Year", fontweight="bold")
axes[0].set_ylabel("Days")

axes[1].bar(national.index, national["hw_events"], color="#e67e22", alpha=0.8)
axes[1].set_title("Mean Heatwave Events per Year", fontweight="bold")
axes[1].set_ylabel("Events")

axes[2].plot(national.index, national["summer_tmax_mean"], "o-", color="#c0392b", linewidth=2)
axes[2].set_title("Mean Summer Tmax", fontweight="bold")
axes[2].set_ylabel("°C")

for ax in axes:
    ax.set_xlabel("Year")
    ax.grid(True, alpha=0.3)
    ax.axvline(x=2022, color="darkred", linestyle="--", alpha=0.4, label="2022")

plt.suptitle("Heatwave Exposure (2012–2022)", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig(fig_dir / "heatwave_timeseries.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Regional heatwave trajectories
name_col = "region_name" if "region_name" in panel.columns else "nuts2_code"

fig, ax = plt.subplots(figsize=(14, 6))
for region, grp in panel.groupby(name_col):
    grp_s = grp.sort_values("year")
    ax.plot(grp_s["year"], grp_s["hw_days"], alpha=0.45, linewidth=1.2, label=region)

ax.set_xlabel("Year")
ax.set_ylabel("Heatwave Days")
ax.set_title("Heatwave Days by Region (2012–2022)", fontweight="bold")
ax.axvline(x=2022, color="darkred", linestyle="--", alpha=0.5, label="2022")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=7, ncol=1)
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(fig_dir / "heatwave_by_region.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 5. Mortality vs. Heat

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

scatter = ax.scatter(
    panel["hw_days"],
    panel["mortality_rate"],
    c=panel["rsvi"],
    cmap="YlOrRd",
    alpha=0.7,
    edgecolors="white",
    linewidths=0.5,
    s=60,
)
plt.colorbar(scatter, ax=ax, label="RSVI (Social Vulnerability)")

# Highlight 2022
yr2022 = panel[panel["year"] == 2022]
ax.scatter(yr2022["hw_days"], yr2022["mortality_rate"],
           edgecolors="black", linewidths=1.5, s=80, facecolors="none", label="2022", zorder=5)

ax.set_xlabel("Heatwave Days (summer)")
ax.set_ylabel("Mortality Rate (per 100,000)")
ax.set_title("Mortality Rate vs. Heatwave Exposure
Coloured by RSVI",
             fontsize=13, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(fig_dir / "mortality_vs_heat.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Mortality time series — national mean
mort_ts = panel.groupby("year")["mortality_rate"].agg(["mean", "std"])

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(mort_ts.index, mort_ts["mean"], color="#2c3e50", alpha=0.8)
ax.errorbar(mort_ts.index, mort_ts["mean"], yerr=mort_ts["std"],
            fmt="none", color="black", capsize=4)
ax.axvline(x=2022, color="darkred", linestyle="--", alpha=0.6, label="2022")
ax.set_xlabel("Year")
ax.set_ylabel("Mortality Rate (per 100,000)")
ax.set_title("National Mean Summer Mortality Rate (2012–2022)", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(fig_dir / "mortality_timeseries.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 6. RSVI Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution across all region-years
axes[0].hist(panel["rsvi"], bins=20, color="#3498db", edgecolor="white", alpha=0.8)
axes[0].set_xlabel("RSVI Score")
axes[0].set_ylabel("Count")
axes[0].set_title("RSVI Distribution (all region-years)", fontweight="bold")
axes[0].grid(True, alpha=0.3)

# Mean RSVI by region (2022)
rsvi_2022 = panel[panel["year"] == 2022].sort_values("rsvi", ascending=True)
name_col = "region_name" if "region_name" in rsvi_2022.columns else "nuts2_code"
axes[1].barh(rsvi_2022[name_col], rsvi_2022["rsvi"], color="#e74c3c", alpha=0.8)
axes[1].set_xlabel("RSVI Score")
axes[1].set_title("RSVI by Region (2022)", fontweight="bold")
axes[1].grid(True, alpha=0.3)

plt.suptitle("Regional Social Vulnerability Index", fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig(fig_dir / "rsvi_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 7. Save Tables

In [ ]:
table_dir = get_path(cfg, "tables")
table_dir.mkdir(parents=True, exist_ok=True)

stats.to_csv(table_dir / "summary_stats.csv")
regional.to_csv(table_dir / "summary_by_region.csv")

print("Tables saved to outputs/tables/")
print("Figures saved to outputs/figures/")
print("
=> Next: Notebook 05 — Panel Regression Analysis")